In [15]:
import re
import json
import requests

URL = "https://detergent-direction-wrongness.ngrok-free.dev/generate"  
headers = {"Authorization": "Bearer 123pass123word"}
sample_input = """
John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAI (2023-Present)
"""
payload = {
    "prompt": sample_input,
    "max_length": 1000
}

res = requests.post(URL, headers=headers, json=payload)
answer = res.json()["response"]

In [16]:
res

<Response [200]>

In [17]:
answer

'\nYou are a smart assistant that extracts information from a user\'s cv.\n\nExtract all information regarding the name of the user, his email, his education, his skills, and his experience.\n\nRespond ONLY in JSON format as follows:\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"full_name": string  // Candidate\'s full name\n\t"email": string  // Candidate\'s email address\n\t"education": List[dict]  // List of educational qualifications, where each item is an object with \'degree\', \'institution\', and \'year\' (as integer)\n\t"skills": List[string]  // List of candidate skills as strings\n\t"experience": List[dict]  // List of work experiences, where each item is an object with \'role\', \'company\', and \'years\'\n}\n```\n\nExample Input:\n"Mohamed Negm - moahamed123@email.com\nEducation: B.Sc. Social Science, KFC, 2019\nSkills: cooking, selling, cleaning\nExperience:\n- Wor

In [18]:
def extract_json_block(text):
    if text is None:
        raise ValueError("API returned no response.")

    # If the response is already plain JSON, return it as-is.
    try:
        json.loads(text)
        return text.strip()
    except (TypeError, ValueError):
        pass

    # Otherwise, look for a fenced ```json ... ``` block.
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    if not matches:
        raise ValueError(f"No JSON block found in the API response: {text[:500]}")

    return matches[-1].strip()

json_text = extract_json_block(answer)
print(json_text)

# Optional: parse it if you want a Python object
parsed = json.loads(json_text)
print(parsed)

{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": [
		{
			"degree": "B.Sc. Computer Science",
			"institution": "MIT",
			"year": 2020
		}
	],
	"skills": [
		"Python",
		"Machine Learning",
		"Data Analysis"
	],
	"experience": [
		{
			"role": "Software Engineer",
			"company": "Google",
			"years": "2020-2023"
		},
		{
			"role": "Data Scientist",
			"company": "OpenAI",
			"years": "2023-Present"
		}
	]
}
{'full_name': 'John Smith', 'email': 'john.smith@email.com', 'education': [{'degree': 'B.Sc. Computer Science', 'institution': 'MIT', 'year': 2020}], 'skills': ['Python', 'Machine Learning', 'Data Analysis'], 'experience': [{'role': 'Software Engineer', 'company': 'Google', 'years': '2020-2023'}, {'role': 'Data Scientist', 'company': 'OpenAI', 'years': '2023-Present'}]}
